<a href="https://colab.research.google.com/github/Akhilesh-2308/MedicalTerm-Simplifier/blob/main/Medical_Term_Simplifier%E2%80%94Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF']     = 'expandable_segments:True'

!pip install --no-cache-dir \
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' \
    pyyaml -q

print('✅ Ready')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 220.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 434.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 288.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 439.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 323.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 314.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 236.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 203.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 189.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 256.6 MB/s eta 0:00:00
✅ Ready


In [2]:
import yaml, pathlib, torch, textwrap
from unsloth         import FastLanguageModel
from google.colab    import userdata
from huggingface_hub import login


def load_demo(config_path='config.yaml'):
    """
    Reads config.yaml, authenticates with HuggingFace,
    and loads the trained LoRA adapter from the Hub.
    Returns (model, tokenizer, cfg) ready for inference.
    """
    p = pathlib.Path(config_path)
    if not p.exists():
        raise FileNotFoundError(f"Upload '{config_path}' via the file panel first.")
    cfg = yaml.safe_load(p.read_text())

    token = userdata.get(cfg['export'].get('hub_token_env', 'HF_TOKEN'))
    os.environ['HF_TOKEN'] = token
    login(token=token, add_to_git_credential=False)

    hub_id    = cfg['export']['hub_model_id']
    model_cfg = cfg['model']
    print(f'Loading model from Hub: {hub_id} ...')

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = hub_id,
        max_seq_length = model_cfg['max_seq_length'],
        dtype          = model_cfg['dtype'],
        load_in_4bit   = model_cfg['load_in_4bit'],
    )
    FastLanguageModel.for_inference(model)

    print(f'✅ Model ready — {hub_id}')
    return model, tokenizer, cfg


model, tokenizer, cfg = load_demo()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model from Hub: Akhilesh-2308/medical-term-simplifier-pipeline ...
==((====))==  Unsloth 2026.4.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/54.6M [00:00<?, ?B/s]

Unsloth 2026.4.6 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ Model ready — Akhilesh-2308/medical-term-simplifier-pipeline


In [3]:
# ── Terms shown during presentation ───────────────────────────
SHOWCASE_TERMS = [
    'Myocardial infarction',
    'Pulmonary embolism',
    'Sepsis',
    'Hyperglycemia',
    'Atrial fibrillation',
]


# ── Core inference function ────────────────────────────────────
def demo(term, instruction=None):
    if instruction is None:
        instruction = cfg['dataset']['instruction_variants'][0]

    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{term}\n\n"
        f"### Response:\n"
    )

    inputs    = tokenizer(prompt, return_tensors='pt', truncation=True).to('cuda')
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            eos_token_id   = tokenizer.eos_token_id,
            pad_token_id   = tokenizer.eos_token_id,
            use_cache      = True,
            **cfg['inference'],
        )

    response = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    ).strip()

    width = 72
    print('\n' + '━' * width)
    print(f'  Term   : {term}')
    print('━' * width)
    for line in response.replace('. ', '.\n').split('\n'):
        for chunk in textwrap.wrap(line.strip(), width=width - 4) or ['']:
            print(f'  {chunk}')
    print('━' * width)

    return response


# ── Main — runs all showcase terms ────────────────────────────
def main():
    print(f'\n  Medical Term Simplifier — Live Demo')
    print(f'  Model : {cfg["export"]["hub_model_id"]}')
    print(f'  Terms : {len(SHOWCASE_TERMS)}\n')

    for i, term in enumerate(SHOWCASE_TERMS, 1):
        print(f'  [{i}/{len(SHOWCASE_TERMS)}]', end='')
        demo(term)

    print('\n  ✅  Demo complete.')


# ── Run ───────────────────────────────────────────────────────
main()


# ── After main(), call demo() on any term on the fly ──────────



  Medical Term Simplifier — Live Demo
  Model : Akhilesh-2308/medical-term-simplifier-pipeline
  Terms : 5

  [1/5]

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Term   : Myocardial infarction
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  A myocardial infarction (MI), commonly known as a heart attack,
  occurs when the blood supply to part of the heart muscle is suddenly
  blocked.
  This causes damage or death to the affected area of the heart
  muscle.
  The most common cause is a build-up of fatty deposits in the
  coronary arteries that supply the heart with oxygen and nutrients.
  These deposits narrow the arteries, reducing blood flow.
  If a clot forms on top of one of these deposits, it can completely
  block the artery.
  Without oxygen, the heart muscle begins to die within minutes.
  Symptoms include chest pain, shortness of breath, sweating, nausea,
  and dizziness.
  Treatment includes restoring blood flow to the heart muscle by
  opening the blocked artery with medication or surgery.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Term   : Pulmonary embolism
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  A pulmonary embolism is a blockage of an artery in the lung.
  It occurs when a blood clot or other material travels from another
  part of the body to the lungs and blocks one of the arteries there.
  The most common cause is a blood clot that forms in a vein in the
  leg (deep vein thrombosis) and breaks off, traveling through the
  bloodstream until it lodges in the lungs.
  Other causes include fat, air bubbles, and tumor cells.
  Symptoms may include chest pain, shortness of breath, coughing up
  blood, and rapid heart rate.
  Diagnosis is based on symptoms, physical exam, and imaging tests
  such as CT scan or ultrasound.
  Treatment includes blood-thinning medications and sometimes surgery
  to remove the clot.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [3/5]

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Term   : Sepsis
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Sepsis is a life-threatening condition that occurs when the body's
  response to an infection causes inflammation throughout the body.
  This can lead to organ damage and failure, and can be fatal if not
  treated promptly and aggressively.
  Symptoms of sepsis include fever, chills, rapid breathing, fast
  heart rate, confusion, and low blood pressure.
  Treatment may involve antibiotics, fluids, oxygen support, and other
  measures to stabilize vital signs and organs.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [4/5]

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Term   : Hyperglycemia
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Hyperglycemia is a condition where the blood sugar level is higher
  than normal.
  The normal range for blood glucose levels is 70-140 mg/dL before
  meals and less than 180 mg/dL after meals.
  Hyperglycemia can be caused by various factors, including insulin
  resistance, pancreatic beta cell dysfunction, stress, infection,
  certain medications, and lifestyle choices such as poor diet and
  lack of exercise.
  Symptoms of hyperglycemia may include increased thirst and
  urination, fatigue, blurred vision, and slow healing wounds.
  If left untreated, it can lead to serious complications such as
  diabetic ketoacidosis, nonketotic hyperosmolar coma, and long-term
  damage to the eyes, nerves, kidneys, and blood vessels.
  Treatment typically involves lifestyle changes such as healthy
  eating habits, regular phy